# Test-Bench: kompletter Evaluationsdurchlauf

Dieses Notebook führt den **gesamten Evaluationsworkflow** phase für Phase aus
und nutzt dabei exakt dieselben Funktionen wie das CLI (`python -m evaluation ...`):

1. **Validieren** (offline) — Korpus, Profile, Experiment strikt prüfen
2. **Planen** (offline) — Manifest + deterministischer Jobplan
3. **Live ausführen** — nur mit explizitem Gate `EXECUTE_LIVE = True`
4. **Checks** — automatische fachliche Prüfungen der Antworten
5. **Bewertung** — neutraler Bogen (Export) + Import ausgefüllter Ratings
6. **Bericht** — summary, Paarvergleiche, `report.md`
7. **Analyse** — Tabellen und Inspektion direkt im Notebook

Regeln (siehe `docs/evaluation_protocol.md` und `evaluation/README.md`):

- Ein Job = ein Aufruf von `POST /api/tutor/start` mit **frischem Chat**,
  direkter Zielstufe und **allen zehn Kontextschaltern**.
- `run --execute-live`-Äquivalent nur in Phase 3 mit gesetztem Gate.
- Kein automatischer Retry; unklare Transportversuche bleiben unklar.
- Demonstrationsläufe (`allow_unverified_cases=true`) sind Werkzeugprüfungen,
  **keine Forschungsergebnisse**.
- Fehlgeschlagene Aufrufe bleiben sichtbar; Nicht-Bewertetes zählt als fehlt.

In [ ]:
# ---------------------------------------------------------------------------
# 0) Konfiguration — an den geplanten Lauf anpassen
# ---------------------------------------------------------------------------
EXPERIMENT_FILE = "evaluation/experiments/pilot.json"    # oder context_core.json
CORPUS_FILE = "evaluation/data/example_cases.jsonl"      # später: cases_research.jsonl
PROFILES_FILE = "evaluation/data/context_profiles.json"
RUN_DIR = "evaluation/runs/testbench-001"
BASE_URL = "http://127.0.0.1:8000"
HTTP_TIMEOUT = 420.0

# Sicherheitsgate: nur dann Phase 3 ausführen, wenn echte Tutor-API-Aufrufe
# (Rate-Limit ~2 Upstream-Versuche/Request, konservatives Budget) erwünscht sind.
EXECUTE_LIVE = False
RESUME = False        # True: offene Jobs fortsetzen (Identität wird geprüft)
RETRY_FAILED = False  # True (manuell!): fehlgeschlagene — nie unklare — erneut senden
RATINGS_CSV = ""      # optional: Pfad zur ausgefüllten Bewertungs-CSV für Phase 5b

In [ ]:
# ---------------------------------------------------------------------------
# Hilfsbausteine (nur stdlib; keine versteckten Netzaufrufe)
# ---------------------------------------------------------------------------
import json
import statistics
from collections import defaultdict
from pathlib import Path

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def print_table(headers, rows):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for index, cell in enumerate(row):
            widths[index] = max(widths[index], len(str(cell)))
    line = " | ".join(str(headers[i]).ljust(widths[i]) for i in range(len(headers)))
    print(line)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(row)))

def banner(title):
    print("")
    print("=" * 72)
    print(title)
    print("=" * 72)

## Phase 1 — Validieren (offline)

Prüft Fallbestand, Profile und Experiment strikt; zeigt zusätzlich, welche
(Fall × Profil)-Zellen datenseitig geeignet sind. Ein fachlicher Lauf erfordert
verifizierte Mathematik (`verification.mathematics_status = verified`) —
Fixtures sind für Werkzeugtests.

In [ ]:
banner("Phase 1: Validieren")
import sys
from evaluation.corpus import load_cases, load_profiles, load_experiment, validate_experiment, case_profile_eligible

cases = load_cases(Path(CORPUS_FILE))
profile_set = load_profiles(Path(PROFILES_FILE))
experiment = load_experiment(Path(EXPERIMENT_FILE))
profile_by_id = profile_set.by_id()

errors = validate_experiment(experiment, cases, profile_set)
if errors:
    print("Experiment unbelegbar:")
    for error in errors:
        print(" -", error)
    raise SystemExit(1)

unverified = [case.case_id for case in cases if not case.is_mathematics_verified()]
print(f"Experiment {experiment.experiment_id} · {len(cases)} Fälle "
      f"({len(cases) - len(unverified)} verifiziert) · Profile: "
      f"{', '.join(experiment.profiles)}")
if unverified:
    print(f"Nicht verifiziert: {', '.join(unverified)}")
print(f"allow_unverified_cases={experiment.allow_unverified_cases} → "
      + ("Demonstrationslauf" if experiment.allow_unverified_cases else "Forschungslauf"))

rows = []
for case in sorted(cases, key=lambda c: c.case_id):
    row = [case.case_id]
    for profile_id in experiment.profiles:
        reason = case_profile_eligible(case, profile_by_id[profile_id],
                                       experiment.allow_unverified_cases)
        row.append("✓" if reason is None else "✗ " + reason.split("(")[0].strip())
    rows.append(row)
print_table(["Fall"] + experiment.profiles, rows)

## Phase 2 — Planen (offline, deterministisch)

Erzeugt Manifest (eingefrorene Hashes, Policy-Snapshot, Budget) und
`plan.jsonl`. Ein nichtleeres Run-Verzeichnis wird bewusst abgewiesen —
neue `run_id` wählen oder `RESUME = True` setzen (dann Phase 3).

In [ ]:
banner("Phase 2: Planen")
from evaluation.runner import create_run, load_plan, load_manifest, PLAN_FILE, GENERATIONS_FILE

run_dir = Path(RUN_DIR)
if run_dir.exists() and any(run_dir.iterdir()):
    manifest_existing = load_manifest(run_dir)
    planned = [job for job in load_plan(run_dir)]
    print(f"Run-Verzeichnis bereits belegt ({run_dir}): "
          f"{len(planned)} Jobs im Plan (Experiment {manifest_existing['experiment_id']}).")
    if not RESUME:
        print("Phase 3 ohne RESUME=True ist dann blockiert; neue run_id verwenden.")
    plan_jobs = planned
else:
    manifest = create_run(
        run_dir=run_dir,
        experiment_path=Path(EXPERIMENT_FILE),
        corpus_path=Path(CORPUS_FILE),
        profiles_path=Path(PROFILES_FILE),
        base_url=BASE_URL,
        http_timeout=HTTP_TIMEOUT,
    )
    plan_jobs = load_plan(run_dir)
    print(f"Manifest + Plan erstellt: {run_dir}")
    print(f"Jobs: {manifest['counts']['jobs']} · Ausschlüsse: {manifest['counts']['exclusions']}"
          f" · Budget: {manifest['max_generations_per_hour']} logische Requests/h")

grid = defaultdict(int)
for job in plan_jobs:
    grid[(job['profile_id'], job['hint_level'], job['block'])] += 1
rows = [[profile_id, level, block, count]
        for (profile_id, level, block), count in sorted(grid.items())]
print_table(["Profil", "Stufe", "Block", "Jobs"], rows)

## Phase 3 — Live-Ausführung (nur mit Gate)

Entspricht `python -m evaluation run --run-dir … --execute-live [--resume]`.
 Versandt je Job genau einen Request mit frischem Chat. Für jeden Versuch
wird **vor** dem Senden ins Journal geschrieben; Abbrüche nach Versand
bleiben als `transport_ambiguous` erhalten.

> Rate-Limit beachten: konservativ `max_generations_per_hour` aus dem
> Experiment (Default 35 logische Requests/h, da der SAIA-Client intern
> bis zu zwei Upstream-Versuche je Request auslösen kann).

In [ ]:
banner("Phase 3: Live-Ausführung")
from evaluation.runner import Runner, GENERATIONS_FILE

if not EXECUTE_LIVE:
    print("Übersprungen: EXECUTE_LIVE=False. Für echte Aufrufe oben auf True "
          "setzen (Rate-Limit und isolierte Tutor-Instanz beachten).")
    stats = None
else:
    runner = Runner(run_dir=run_dir)
    stats = runner.run(execute_live=True, resume=RESUME, retry_failed=RETRY_FAILED)
    print_table(
        ["Kennzahl", "Wert"],
        [[key, value] for key, value in stats.items()],
    )

## Phase 4 — Automatische Checks

Sieben Check-Klassen je erfolgreicher Antwort (Identität, Kontextoptionen,
Prompt-Pflichtfelder, Lösungs-Guard, Wortzahl, Stufennennung, Endlösungs-
Offenlegung). `fail` ist ein Befund — keine automatische Nichtwertung; 
`inconclusive` zählt nie als bestanden.

In [ ]:
banner("Phase 4: Checks")
from evaluation.checks import run_checks_for_run

if not (Path(RUN_DIR) / GENERATIONS_FILE).exists():
    print("Übersprungen: noch keine Generierungen im Run-Verzeichnis "
          f"({GENERATIONS_FILE} fehlt).")
else:
    check_result = run_checks_for_run(Path(RUN_DIR))
    print(f"Checks: {check_result['records']} Datensätze · "
          f"{check_result['failed']} fehlgeschlagen · {check_result['path']}")
    checks_all = read_jsonl(Path(RUN_DIR) / "checks.jsonl")
    fails = [record for record in checks_all if record.get("status") == "fail"]
    if fails:
        grouped = defaultdict(int)
        for record in fails:
            grouped[(record["check_id"], record.get("job_id"))] += 1
        print_table(["Check", "Job", "Anzahl"],
                    [[check_id, job_id, count]
                     for (check_id, job_id), count in sorted(grouped.items())])
    else:
        print("Keine fehlgeschlagenen Checks.")

## Phase 5 — Bewertungsexport (blind)

Erzeugt den neutralen Bewertungsbogen **ohne Modell-/Profilspalten** plus
`review_mapping.json` (nicht an Bewerter weitergeben). Spalten und Anker:
`evaluation/rubric.md`.

In [ ]:
banner("Phase 5: Bewertungsexport")
from evaluation.report import export_review_packet

if not (Path(RUN_DIR) / GENERATIONS_FILE).exists():
    print("Übersprungen: keine Generierungen vorhanden.")
else:
    export = export_review_packet(Path(RUN_DIR))
    print(f"Bogen: {export['path']} · {export['packet_rows']} Antworten")
    print("Weiter: CSV ausfüllen (rubric.md), dann Phase 5b mit RATINGS_CSV.")

## Phase 5b — Bewertungsimport (optional)

Setzt `RATINGS_CSV` oben, wenn eine ausgefüllte Bewertungsdatei vorliegt.
Doppelbewertungen derselben (`review_id`, `rater_id`)-Kombination werden
nicht dupliziert; ungültige Werte erzeugen Fehlermeldungen statt stiller
Korrekturen.

In [ ]:
banner("Phase 5b: Bewertungsimport")
from evaluation.report import import_review_ratings

if not RATINGS_CSV:
    print("Übersprungen: RATINGS_CSV ist leer (optionaler Schritt).")
else:
    import_result = import_review_ratings(Path(RUN_DIR), Path(RATINGS_CSV))
    print(f"Importiert: {import_result['imported']} · "
          f"gespeichert: {import_result['records']} · "
          f"Fehler: {len(import_result['errors'])}")
    for error in import_result["errors"][:20]:
        print(" -", error)

## Phase 6 — Bericht (offline aus Artefakten)

Erzeugt `derived/summary.csv`, `derived/paired_comparisons.csv` und
`derived/report.md`. Lösungen auf Stufe 4 sind erlaubt; der Bericht trennt
`complete_solution_present` von `prohibited_disclosure`.

In [ ]:
banner("Phase 6: Bericht")
from evaluation.report import build_report

if not (Path(RUN_DIR) / GENERATIONS_FILE).exists():
    print("Übersprungen: keine Generierungen vorhanden.")
else:
    report_result = build_report(Path(RUN_DIR))
    print("Abdeckung:", json.dumps(report_result["coverage"], ensure_ascii=False))
    print()
    print((run_dir / "derived" / "report.md").read_text(encoding="utf-8"))

## Phase 7 — Analyse im Notebook

Nur Live-Datensätze (`execution_source == 'live_tutor_api'`); Mock/Fixture
zählt nicht. Fehlversuche werden nicht still entfernt.

In [ ]:
banner("Phase 7: Durchlauf & Dauer")
generations = [
    record for record in read_jsonl(Path(RUN_DIR) / GENERATIONS_FILE)
    if record.get("execution_source") == "live_tutor_api"
]
outcomes = defaultdict(int)
durations = []
for record in generations:
    outcomes[record.get("outcome", "unbekannt")] += 1
    if record.get("duration_ms") is not None:
        durations.append(record["duration_ms"])
rows = [[outcome, count] for outcome, count in sorted(outcomes.items())]
print_table(["Ausgang", "Anzahl"], rows)
if durations:
    durations.sort()
    p95 = durations[min(len(durations) - 1, int(0.95 * len(durations)))]
    print(f"Dauer (ms): median {int(statistics.median(durations))} · p95 {p95}")

failed_rows = [record for record in generations if record.get("outcome") != "success"]
print(f"\nFehlgeschlagene/unbekannte Versuche: {len(failed_rows)}")
for record in failed_rows[:20]:
    print(" ", record.get("job_id"), "→", record.get("outcome"),
          (record.get("safe_error") or "")[:90])

In [ ]:
banner("Phase 7: Offenlegung & Prüfbefunde je Profil/Stufe")
manifest_path = Path(RUN_DIR) / "manifest.json"
if not manifest_path.exists():
    print("Übersprungen: kein Manifest (Phase 2 noch nicht ausgeführt).")
else:
    checks_by_attempt = defaultdict(list)
    for check in read_jsonl(Path(RUN_DIR) / "checks.jsonl"):
        checks_by_attempt[check["attempt_id"]].append(check)

    hint_policy = json.loads(manifest_path.read_text(encoding="utf-8")).get("hint_policy", {})
    grouped = defaultdict(list)
    for record in generations:
        if record.get("outcome") == "success":
            grouped[(record.get("profile_id"), record.get("hint_level"))].append(record)

    rows = []
    for (profile_id, level), records in grouped.items():
        fails = defaultdict(int)
        present = prohibited = 0
        for record in records:
            for check in checks_by_attempt.get(record["attempt_id"], []):
                if check.get("status") == "fail":
                    fails[check["check_id"]] += 1
                if check["check_id"] == "final_answer_disclosure":
                    if check.get("evidence", {}).get("present"):
                        present += 1
                    if check.get("status") == "fail":
                        prohibited += 1
        level_policy = hint_policy.get(str(level), {})
        rows.append([profile_id, level, len(records), sum(fails.values()),
                     present, prohibited,
                     f"Endlösung erlaubt: {bool(level_policy.get('include_final_answer'))}"])
    print_table(["Profil", "Stufe", "n", "Prüffehler", "Lösung vorhanden",
                 "unzulässig", "Policystatus"], rows)
    print("\\nHinweis: fehlende Treffer beweisen keine Abwesenheit von "
          "Lösungsverrat (symbolische Tiefe begrenzt, sympy optional).")

In [ ]:
banner("Phase 7: Paarvergleiche gegen 'base' (Hilfreichkeit)")
ratings_by_attempt = defaultdict(list)
for rating in read_jsonl(Path(RUN_DIR) / "reviews" / "ratings.jsonl"):
    ratings_by_attempt[rating["attempt_id"]].append(rating)

def median_helpfulness(record):
    values = [r["ratings"].get("hilfreichkeit_naechster_schritt")
              for r in ratings_by_attempt.get(record["attempt_id"], [])
              if r["ratings"].get("hilfreichkeit_naechster_schritt") is not None]
    return statistics.median(values) if values else None

by_key = defaultdict(dict)
for record in generations:
    if record.get("outcome") == "success":
        key = (record.get("case_id"), record.get("hint_level"),
               record.get("repetition"))
        by_key[key][record.get("profile_id")] = record

deltas = defaultdict(list)
for (case_id, level, repetition), profiles_map in by_key.items():
    base_value = median_helpfulness(profiles_map.get("base"))
    for profile_id, record in profiles_map.items():
        if profile_id == "base" or base_value is None:
            continue
        value = median_helpfulness(record)
        if value is not None:
            deltas[profile_id].append(value - base_value)

if any(deltas.values()):
    rows = [[profile_id, f"{statistics.median(values):+.1f}", len(values)]
            for profile_id, values in sorted(deltas.items())]
    print_table(["Profil", "Δ Median", "n"], rows)
else:
    print("Noch keine Bewertungen — Phase 5b ausführen oder Werte ergänzen.")

In [ ]:
banner("Phase 7: Beispielbetrachter")
show_profiles = sorted({record.get("profile_id") for record in generations
                        if record.get("outcome") == "success"})
shown = set()
for record in generations:
    profile_id = record.get("profile_id")
    if record.get("outcome") != "success" or profile_id in shown:
        continue
    shown.add(profile_id)
    returned = record.get("returned") or {}
    print(f"\n=== {profile_id} · Stufe {record.get('hint_level')} · "
          f"Fall {record.get('case_id')} (attempt {record.get('attempt_id')})")
    print((returned.get("hint") or "")[:700])

## Grenzen & Verweise

- Tokenverbrauch, Upstream-Versuchszahl, `finish_reason`: laut Manifest
  **unbekannt** — die Tutor-API übermittelt sie nicht; der Runner schätzt sie nicht.
- `POST /api/tutor/start` validiert Diagnosen nicht fachlich: die
  Beweislast liegt im Korpus (Provenienz + Belege).
- Kernlauf (`context_core.json`) erfordert den verifizierten Korpus
  `data/cases_research.jsonl` (aus STACK-/Moodle-Exporten, Phase-AP7).
- Zentrale Doku: `evaluation/README.md`, `docs/evaluation_protocol.md`,
  `evaluation/rubric.md`. Reines Analyse-Notebook (ohne Ausführung):
  `notebooks/auswertung.ipynb`.